# Fase 3 — Análisis exploratorio (Flujo B, parte 2)

| Campo | Valor |
|---|---|
| **Rol líder** | Analista de Datos |
| **Entrada** | `data/processed/chembl_clean.csv` |
| **Salida** | Figuras en `outputs/chembl/figures/` + tablas de correlación |
| **Doc de fase** | [`docs/analisis_proyecto/fases/fase3_eda.md`](../../docs/analisis_proyecto/fases/fase3_eda.md) |

Secciones 1 y 3 del Flujo B: EDA y correlaciones.

> **Prerequisito:** ejecutar `fase2_limpieza.ipynb` para generar `chembl_clean.csv`.


## 0. Configuración


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd().parent.parent if Path.cwd().name == "proyecto analisis de datos" else (
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analisis_proyecto.chembl_preprocessing import (
    NUMERIC_DESCRIPTOR_COLS,
    correlation_with_target,
    get_available_feature_cols,
    load_bioactivity,
    summary_statistics,
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

CLEAN_CSV = ROOT / "data" / "processed" / "chembl_clean.csv"
FIG_DIR = ROOT / "outputs" / "chembl" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert CLEAN_CSV.exists(), (
    f"No se encontró {CLEAN_CSV}. Ejecuta primero fase2_limpieza.ipynb"
)

df_clean = load_bioactivity(CLEAN_CSV)
df = df_clean  # la sección 1 del notebook original usa el alias `df`
print(f"Dataset limpio: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas")
print(f"Compuestos: {df_clean['compound_name'].nunique()}")


## 1. Análisis preliminar

Medidas de tendencia central, distribuciones y conteos categóricos.


In [ ]:
eda_numeric = [c for c in NUMERIC_DESCRIPTOR_COLS if c in df.columns]
display(df[eda_numeric].describe().T)

stats_table = summary_statistics(df, eda_numeric)
display(stats_table.round(4))


In [ ]:
for col, title in [
    ("pchembl_value", "pChEMBL value"),
    ("mw_freebase", "Peso molecular"),
    ("alogp", "LogP (ALogP)"),
]:
    if col not in df.columns:
        continue
    fig, ax = plt.subplots()
    df[col].dropna().hist(bins=40, ax=ax, edgecolor="white", color="#4C72B0")
    ax.set_title(f"Distribución: {title}")
    ax.set_xlabel(col)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"hist_{col}.png", bbox_inches="tight")
    plt.show()


In [ ]:
for col in ["pchembl_value", "alogp"]:
    if col not in df.columns:
        continue
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=df, x="family", y=col, ax=ax)
    ax.set_title(f"{col} por familia química")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"box_{col}_by_family.png", bbox_inches="tight")
    plt.show()

for cat_col in ["activity_class", "assay_type", "standard_type", "target_type"]:
    if cat_col not in df.columns:
        continue
    print(f"\n=== {cat_col} ===")
    display(df[cat_col].value_counts(dropna=False).to_frame("n"))


## 3. Correlación

Pearson y Spearman vs `pchembl_value`; heatmap y pairplot de variables top.


In [ ]:
corr_cols = [c for c in get_available_feature_cols(df_clean) if c in df_clean.columns]
corr_table = correlation_with_target(df_clean, columns=corr_cols)
display(corr_table.round(4))

if len(corr_cols) >= 2:
    corr_matrix = df_clean[corr_cols + ["pchembl_value"]].corr(method="pearson")
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Correlación de Pearson (descriptores moleculares)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "correlation_heatmap.png", bbox_inches="tight")
    plt.show()

top_vars = corr_table.head(5)["variable"].tolist() if not corr_table.empty else corr_cols[:4]
pair_cols = list(dict.fromkeys(top_vars + ["pchembl_value"]))
if len(pair_cols) >= 2:
    g = sns.pairplot(df_clean[pair_cols].dropna(), diag_kind="hist", corner=True)
    g.fig.suptitle("Scatter matrix — variables top vs pChEMBL", y=1.02)
    g.savefig(FIG_DIR / "correlation_pairplot.png", bbox_inches="tight")
    plt.show()


---
*Fase anterior:* [`fase2_limpieza.ipynb`](fase2_limpieza.ipynb)  
*Siguiente fase:* [`fase4_modelado.ipynb`](fase4_modelado.ipynb)
